In [ ]:
"""
Robot → InfluxDB Logger — Neurapy
"""
import time
import influxdb_client
import os
import subprocess

from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import WriteOptions

# ── Avvio container ──────────────────────────────────────────────────────────
subprocess.run(["docker", "start", "mio_influxdb"], capture_output=True)
time.sleep(5)

# ── CONFIG ───────────────────────────────────────────────────────────────────
INFLUX_URL    = "http://localhost:8086"
INFLUX_TOKEN  = os.environ.get("INFLUXDB_TOKEN")
if not INFLUX_TOKEN:
    raise ValueError("INFLUXDB_TOKEN non trovato nelle variabili d'ambiente")
INFLUX_ORG    = "Polimi"
INFLUX_BUCKET = "KAWASAKI"

POLL_INTERVAL      = 0.02   # secondi (0.02 = 50 Hz)
VELOCITY_THRESHOLD = 0.01

# Measurement dedicato alla persistenza del contatore traiettorie.
COUNTER_MEASUREMENT = "_trajectory_counter"

# ── Robot ────────────────────────────────────────────────────────────────────
try:
    from neurapy.robot import Robot
    robot = Robot()
    print("[INFO] Connesso al robot reale.")
except BaseException:
    print("[INFO] neurapy non trovato — modalità DEMO attiva.")

# NOTA: joint_velocities è rimosso da METRICS — viene letto separatamente
# nel loop principale e riutilizzato come punto dati, senza doppia chiamata.
METRICS = [
    ("joint_angles",       robot.get_current_joint_angles_with_timestamp),
    ("joint_torques",      robot.get_current_joint_torques_with_timestamp),
    ("load_side_encoder",  robot.get_current_load_side_encoder_values_with_timestamp),
    ("motor_side_encoder", robot.get_current_motor_side_encoder_values_with_timestamp),
]

# ── InfluxDB client ──────────────────────────────────────────────────────────
# Scrittura asincrona con batch: il client accumula i punti e li invia
# ogni BATCH_INTERVAL ms oppure quando raggiunge BATCH_SIZE punti.
# Il loop non si blocca ad aspettare la risposta HTTP di InfluxDB.

client    = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
write_api = client.write_api(write_options=WriteOptions(
    batch_size=50,
    flush_interval=500,
    jitter_interval=0
))
query_api = client.query_api()


# ── Funzione per RECUPERARE ULTIMO ID traiettoria SALVATO ─────────────────────────────

def get_last_trajectory_id() -> int:
    """
    Legge da InfluxDB l'ultimo trajectory_id salvato nel measurement COUNTER_MEASUREMENT. 
    Strategia: query Flux che prende l'ultimo valore del field 'last_id' nel measurement dedicato.
    """
    flux = f'''
from(bucket: "{INFLUX_BUCKET}")
  |> range(start: -100y)
  |> filter(fn: (r) => r._measurement == "{COUNTER_MEASUREMENT}")
  |> filter(fn: (r) => r._field == "last_id")
  |> last()
'''
    try:
        tables = query_api.query(flux, org=INFLUX_ORG)
        for table in tables:
            for record in table.records:
                return int(record.get_value())
    except Exception as e:
        print(f"[WARN] Impossibile leggere l'ultimo ID da InfluxDB: {e}")
    return 0


def save_trajectory_id(traj_id: int) -> None:
    """
    Scrive il nuovo trajectory_id nel measurement.
    """
    p = (Point(COUNTER_MEASUREMENT).field("last_id", traj_id))
    try:
        write_api.write(bucket=INFLUX_BUCKET, record=p)
    except Exception as e:
        print(f"[WARN] Impossibile salvare l'ID traiettoria su InfluxDB: {e}")


# ── Inizializzazione stato traiettoria ────────────────────────────────────────

last_id      = get_last_trajectory_id()
traj_id      = last_id
was_moving   = False
traj_active  = False

print(f"[INFO] Ultimo trajectory_id trovato su InfluxDB: {last_id}")
print(f"[INFO] Prossima traiettoria partirà con ID: {last_id + 1}")
print(f"[INFO] Scrittura avviata (Ctrl+C per fermare)\n")

# ── Loop principale ───────────────────────────────────────────────────────────
try:
    while True:
        t0 = time.monotonic()
        points = []

        # Leggi le velocità UNA SOLA VOLTA: serve sia per rilevare il movimento
        # che per costruire il punto dati — nessuna chiamata duplicata.
        velocities, vel_ts = robot.get_current_joint_velocities_with_timestamp()
        moving = any(abs(v) > VELOCITY_THRESHOLD for v in velocities)

        # ── Rilevamento inizio nuova traiettoria ──────────────────────────────
        if moving and not was_moving:
            traj_id     = last_id + 1
            last_id     = traj_id
            traj_active = True
            save_trajectory_id(traj_id)
            print(f"[{time.strftime('%H:%M:%S')}] ▶ Nuova traiettoria: ID = {traj_id}")

        elif not moving and was_moving:
            print(f"[{time.strftime('%H:%M:%S')}] ■ Fine traiettoria ID = {traj_id}")
            traj_active = False

        was_moving = moving

        # ── Scrittura dati ────────────────────────────────────────────────────
        if moving:
            # Aggiungi il punto delle velocità (già letto sopra)
            p_vel = (Point("joint_velocities").tag("trajectory_id", str(traj_id)).time(int(vel_ts), "us"))
            for i, v in enumerate(velocities):
                p_vel = p_vel.field(f"j{i+1}", float(v))
            points.append(p_vel)

            # Aggiungi gli altri measurement
            for name, fn in METRICS:
                values, ts = fn()
                p = (Point(name)
                     .tag("trajectory_id", str(traj_id))
                     .time(int(ts), "us"))
                for i, v in enumerate(values):
                    p = p.field(f"j{i+1}", float(v))
                points.append(p)

            # Scrittura non bloccante: i punti entrano nel buffer interno del client; 
            # il flush avviene in background ogni BATCH_INTERVAL ms.
            write_api.write(bucket=INFLUX_BUCKET, record=points)
            print(f"[{time.strftime('%H:%M:%S')}] accodati {len(points)} measurement "
                  f"(traj={traj_id})")
        else:
            print(f"[{time.strftime('%H:%M:%S')}] fermo (non scrivo)")

        time.sleep(max(0, POLL_INTERVAL - (time.monotonic() - t0)))

except KeyboardInterrupt:
    print("\n[INFO] Stop.")
finally:
    # Il close() svuota il buffer residuo prima di chiudere la connessione.
    client.close()
